# تست اصلاح‌شده: بررسی واقعی فرمت s2u_map و استخراج بردار جابجایی هر Image

## چرا این نسخه؟
در تست قبلی، `s2u_map[9] = 9` بود در حالی که سلول واحد فقط ۸ اتم داشت (اندیس معتبر ۰-۷).
این یعنی فرض من درباره‌ی فرمت `s2u_map` اشتباه بود. در این نسخه، به‌جای فرض کردن، خودِ
attributeهای موجود روی آبجکت `supercell` را مستقیماً بررسی و چاپ می‌کنیم تا فرمت واقعی
مشخص شود — هم `s2u_map` و هم جایگزین‌های احتمالی (`get_supercell_to_unitcell_map`،
یا استخراج مستقیم از روی تقسیم باقی‌مانده‌ی فاصله‌ی کارتزی).

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
test_formula = common[0]
print(f"ماده‌ی تستی: {test_formula}")

with open(band_yaml_files[test_formula]) as f:
    data = yaml.safe_load(f)

lattice = np.array(data['lattice'])
symbols = [p['symbol'] for p in data['points']]
frac_coords = np.array([p['coordinates'] for p in data['points']])
n_atoms = len(symbols)
print(f"n_atoms (سلول واحد): {n_atoms}")

unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

fc = parse_FORCE_CONSTANTS(str(fc_files[test_formula]))
n_supercell = fc.shape[1]
n_images = n_supercell // n_atoms
print(f"n_supercell: {n_supercell} -> n_images: {n_images}")

def guess_dim(n):
    for a in range(1, n+1):
        if n % a == 0:
            rem = n // a
            for b in range(1, rem+1):
                if rem % b == 0:
                    c = rem // b
                    return (a, b, c)
    return (n, 1, 1)

dim_guess = guess_dim(n_images)
print(f"ابعاد سوپرسل (حدس): {dim_guess}")

phonon = Phonopy(unitcell, supercell_matrix=[[dim_guess[0],0,0],[0,dim_guess[1],0],[0,0,dim_guess[2]]])
phonon.force_constants = fc
print(f"✅ Phonopy ساخته شد. n_atoms سوپرسل: {len(phonon.supercell.symbols)}")

## بررسی مستقیم همه‌ی attributeهای مرتبط با نگاشت سوپرسل↔سلول‌واحد

به‌جای فرض کردن، هرچه phonopy در اختیار می‌گذارد را چاپ می‌کنیم.

In [ ]:
supercell = phonon.supercell

# لیست تمام attributeهای مرتبط با map
candidate_attrs = ['s2u_map', 'u2s_map', 'u2u_map', 'sc2u_map', 'get_supercell_to_unitcell_map']
for attr in candidate_attrs:
    if hasattr(supercell, attr):
        val = getattr(supercell, attr)
        if callable(val):
            try:
                val = val()
            except Exception as e:
                val = f"<error calling: {e}>"
        print(f"{attr}: type={type(val)}")
        try:
            print(f"   مقدار: {np.array(val)}")
            print(f"   شکل: {np.array(val).shape}, min={np.min(val)}, max={np.max(val)}")
        except Exception as e:
            print(f"   (نمی‌توان به آرایه تبدیل کرد: {e})")
    else:
        print(f"{attr}: ❌ وجود ندارد")
    print()

## روش جایگزین و مطمئن‌تر: استخراج بردار جابجایی مستقیماً از روی موقعیت‌های کارتزی

به‌جای تکیه بر `s2u_map` (که فرمتش نامشخص شد)، یک روش مستقل و مطمئن:
هر اتم سوپرسل با شماره‌ی `i`، در ساختار دوره‌ای (periodic) باید با یکی از اتم‌های سلول
واحد، به‌اضافه‌ی یک بردار شبکه‌ای صحیح (R)، منطبق شود:

```
position_supercell[i] = position_unitcell[j] + R   (برای یک j و یک R صحیح)
```

برای پیدا کردن `j` صحیح، کافی است برای هر اتم سوپرسل، نزدیک‌ترین اتم سلول واحد را از نظر
**باقیمانده‌ی fractional coordinate** (یعنی `frac % 1`) پیدا کنیم — چون این مقدار باید
دقیقاً با fractional coordinate سلول واحد یکی باشد (با تقریب عددی).

In [ ]:
# موقعیت‌های فرکشنال سوپرسل (نسبت به لاتیس سوپرسل)
sc_frac = supercell.scaled_positions  # (n_supercell, 3), در مبنای لاتیس سوپرسل
sc_cart = supercell.positions          # (n_supercell, 3), کارتزی واقعی (Å)

uc_cart = unitcell.positions           # (n_atoms, 3), کارتزی واقعی (Å)
uc_lattice = unitcell.cell             # (3,3)

print(f"sc_frac شکل: {sc_frac.shape}")
print(f"sc_cart شکل: {sc_cart.shape}")
print(f"uc_cart شکل: {uc_cart.shape}")
print(f"چند sc_cart اول:\n{sc_cart[:5]}")
print(f"\nuc_cart کامل:\n{uc_cart}")

In [ ]:
# برای هر اتم سوپرسل: پیدا کردن نزدیک‌ترین اتم سلول واحد با در نظر گرفتن تصاویر دوره‌ای (PBC)
# با جستجو روی بردارهای شبکه‌ای R در یک محدوده‌ی کوچک اطراف (برای اطمینان از پیدا کردن match دقیق)

uc_lat_vectors = uc_lattice  # 3x3: ردیف‌ها بردارهای شبکه‌ی سلول واحد هستند

# محدوده‌ی جستجو برای ضرایب صحیح بردار شبکه (-3..3 برای اطمینان، چون سوپرسل تا 3x3x3 رفته)
search_range = range(-4, 5)

unitcell_idx_for_sc_atom = np.full(len(sc_cart), -1, dtype=int)
R_vector_for_sc_atom = np.zeros((len(sc_cart), 3))

for i, pos_i in enumerate(sc_cart):
    best_match = None
    best_dist = np.inf
    for j, pos_j in enumerate(uc_cart):
        for n1 in search_range:
            for n2 in search_range:
                for n3 in search_range:
                    R = n1*uc_lat_vectors[0] + n2*uc_lat_vectors[1] + n3*uc_lat_vectors[2]
                    candidate_pos = pos_j + R
                    dist = np.linalg.norm(pos_i - candidate_pos)
                    if dist < best_dist:
                        best_dist = dist
                        best_match = (j, R)
    unitcell_idx_for_sc_atom[i] = best_match[0]
    R_vector_for_sc_atom[i] = best_match[1]
    if best_dist > 0.05:
        print(f"⚠️ اتم {i}: بهترین تطابق با خطای {best_dist:.4f} Å (بزرگ، مشکل‌دار)")

print("✅ نگاشت کامل شد (ممکن است کند باشد، فقط برای ۱ ماده اجرا می‌شود)")
print(f"\nunitcell_idx_for_sc_atom (10 مقدار اول): {unitcell_idx_for_sc_atom[:10]}")
print(f"محدوده مقادیر: min={unitcell_idx_for_sc_atom.min()}, max={unitcell_idx_for_sc_atom.max()} (باید بین 0 و {n_atoms-1} باشد)")

## گروه‌بندی به تصاویر یکتا بر اساس بردار R (نه فرض ترتیب)

این‌بار به‌جای فرض کردن که اتم‌های هر تصویر پشت‌سرهم می‌آیند، مستقیماً بر اساس بردار `R`
یکسان گروه‌بندی می‌کنیم — این روش به ترتیب اتم‌ها در آرایه‌ی phonopy حساس نیست.

In [ ]:
R_rounded = np.round(R_vector_for_sc_atom, decimals=3)
unique_Rs, inverse_idx, counts = np.unique(R_rounded, axis=0, return_inverse=True, return_counts=True)

print(f"تعداد بردارهای R یکتا: {len(unique_Rs)} (انتظار: {n_images})")
print(f"\nهمه‌ی بردارهای R یکتا:")
for r, c in zip(unique_Rs, counts):
    print(f"  R={r}, تعداد اتم با این R: {c} (انتظار: {n_atoms})")

# بررسی: آیا هر گروه دقیقاً n_atoms عضو دارد؟
all_groups_correct_size = all(c == n_atoms for c in counts)
print(f"\n✅ همه‌ی گروه‌ها اندازه‌ی درست ({n_atoms}) دارند: {all_groups_correct_size}")
print(f"✅ تعداد گروه‌ها برابر n_images است: {len(unique_Rs) == n_images}")

## نتیجه‌گیری نهایی این تست

اگر در خروجی بالا:
- تعداد بردارهای R یکتا == `n_images` (مثلاً ۹)
- هر گروه دقیقاً `n_atoms` عضو دارد (مثلاً ۸)
- هیچ هشدار "خطای بزرگ" در سلول قبلی چاپ نشده باشد

آن‌وقت روش "نزدیک‌ترین تطابق با بردار شبکه‌ای صحیح" قابل‌اعتماد است و در Notebook 18 از
همین `unique_Rs` (به‌عنوان feature ورودی واقعی هر تصویر، به‌جای embedding دلخواه) استفاده
خواهیم کرد.

**لطفاً کل خروجی این نوت‌بوک (همه‌ی پرینت‌ها، به‌خصوص بخش `s2u_map`/`u2s_map` و بخش
بردارهای R یکتا) را برای من بفرستید.**